In [ ]:
import pandas as pd
import requests
import mlflow
from dotenv import load_dotenv
import os

load_dotenv()
#test

API_KEY_S3 = os.environ["AWS_ACCESS_KEY_ID"]
API_SECRET_KEY_S3 = os.environ["AWS_SECRET_ACCESS_KEY"]

In [ ]:
#---MLFlow params
os.environ["APP_URI"] = "https://renergies99lead-mlflow.hf.space/"
EXPERIMENT_NAME = "all_columns_models"

mlflow.set_tracking_uri(os.environ["APP_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

In [16]:
run = 'dd977154007f474993f35e5c5d8361b9'
logged_model = f'runs:/{run}/pipeline_model'
logged_model

'runs:/dd977154007f474993f35e5c5d8361b9/pipeline_model'

In [15]:
loaded_model = mlflow.pyfunc.load_model('runs:/dd977154007f474993f35e5c5d8361b9/model')
artifact_uri = mlflow.get_run('dd977154007f474993f35e5c5d8361b9').info.artifact_uri
errors = mlflow.artifacts.load_dict(artifact_uri + "/error.json")

2025/11/21 10:56:21 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.5.1, required: mlflow==3.6.0)
 - numpy (current: 2.3.1, required: numpy==2.0.1)
 - pandas (current: 2.3.2, required: pandas==2.3.3)
 - psutil (current: 5.9.0, required: psutil==7.0.0)
 - pyarrow (current: 21.0.0, required: pyarrow==19.0.1)
 - scikit-learn (current: 1.7.1, required: scikit-learn==1.7.2)
 - scipy (current: 1.16.0, required: scipy==1.16.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/21 10:56:21 WARNING mlflow.pyfunc: The version of Python that the model was saved in, `Python 3.12.3`, differs from the version of Python that is currently running, `Python 3.13.5`, and may be incompatible
c:\Users\dubos\miniforge3\envs\JedhaTraining\Lib\site-packages\sklearn\b

In [ ]:
input_all = mlflow.artifacts.load_dict('s3://renergies99-lead-mlflow/5/models/m-1575415fcbb140599335c429880392cb/artifacts/input_example.json')
input_col = input_all['columns']
input_data = input_all['data']

In [12]:
input.dtypes

Moulins_temp                float64
Moulins_humidity              int64
Moulins_dew_point           float64
Moulins_clouds                int64
Moulins_wind_speed          float64
Moulins_wind_deg              int64
Aurillac_temp               float64
Aurillac_humidity             int64
Aurillac_dew_point          float64
Aurillac_clouds               int64
Aurillac_wind_speed         float64
Aurillac_wind_deg             int64
Saint-Étienne_temp          float64
Saint-Étienne_humidity        int64
Saint-Étienne_dew_point     float64
Saint-Étienne_clouds          int64
Saint-Étienne_wind_speed    float64
Saint-Étienne_wind_deg        int64
Annecy_temp                 float64
Annecy_humidity               int64
Annecy_dew_point            float64
Annecy_clouds                 int64
Annecy_wind_speed           float64
Annecy_wind_deg               int64
Nyons_temp                  float64
Nyons_humidity                int64
Nyons_dew_point             float64
Nyons_clouds                

In [13]:
input = pd.DataFrame(input_data, columns=input_col)
input.iloc[1].to_json(orient='index')


'{"Moulins_temp":15.65,"Moulins_humidity":94.0,"Moulins_dew_point":14.69,"Moulins_clouds":100.0,"Moulins_wind_speed":2.66,"Moulins_wind_deg":229.0,"Aurillac_temp":12.82,"Aurillac_humidity":91.0,"Aurillac_dew_point":11.39,"Aurillac_clouds":100.0,"Aurillac_wind_speed":3.09,"Aurillac_wind_deg":220.0,"Saint-\\u00c9tienne_temp":18.13,"Saint-\\u00c9tienne_humidity":55.0,"Saint-\\u00c9tienne_dew_point":8.96,"Saint-\\u00c9tienne_clouds":0.0,"Saint-\\u00c9tienne_wind_speed":4.12,"Saint-\\u00c9tienne_wind_deg":210.0,"Annecy_temp":18.53,"Annecy_humidity":72.0,"Annecy_dew_point":13.39,"Annecy_clouds":0.0,"Annecy_wind_speed":1.54,"Annecy_wind_deg":200.0,"Nyons_temp":23.33,"Nyons_humidity":58.0,"Nyons_dew_point":14.61,"Nyons_clouds":100.0,"Nyons_wind_speed":2.16,"Nyons_wind_deg":203.0,"10cm":180.0,"K index Planetary":2.375,"Moulins_day_length":14.7358333333}'

In [ ]:
url = 'https://renergies99lead-api-renergy-lead.hf.space/predict'

pred = requests.post(url, input.iloc[1].to_json(orient='index'))
pred

<Response [200]>

In [92]:
pred.content

b'{"time":"2025-03-02","prediction":39.06071447532446,"error":[6.4335056155]}'

In [69]:
data = pd.read_json(input.iloc[1].to_json(orient='index'), orient='index')
data

C:\Users\dubos\AppData\Local\Temp\ipykernel_24848\1785892946.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  data = pd.read_json(input.iloc[1].to_json(orient='index'), orient='index')


,0
Moulins_temp,15.63
Moulins_humidity,65.00
Moulins_dew_point,9.08
Moulins_clouds,92.00
Moulins_wind_speed,2.52
Moulins_wind_deg,180.00
Aurillac_temp,13.99
Aurillac_humidity,67.00
Aurillac_dew_point,7.97
Aurillac_clouds,100.00


In [52]:
model_uri = 'runs:/51febeee1eb74612b4492dd0e4c8169e/pipeline_model'
# test_model = mlflow.sklearn.load_model(model_uri)
loaded_model = mlflow.pyfunc.load_model(model_uri)

loaded_model.predict(input.iloc[1].to_json())

2025/11/20 16:16:22 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.5.1, required: mlflow==3.6.0)
 - numpy (current: 2.3.1, required: numpy==2.0.1)
 - pandas (current: 2.3.2, required: pandas==2.3.3)
 - psutil (current: 5.9.0, required: psutil==7.0.0)
 - pyarrow (current: 21.0.0, required: pyarrow==19.0.1)
 - scikit-learn (current: 1.7.1, required: scikit-learn==1.7.2)
 - scipy (current: 1.16.0, required: scipy==1.16.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/20 16:16:22 WARNING mlflow.pyfunc: The version of Python that the model was saved in, `Python 3.12.3`, differs from the version of Python that is currently running, `Python 3.13.5`, and may be incompatible
c:\Users\dubos\miniforge3\envs\JedhaTraining\Lib\site-packages\sklearn\b

MlflowException: Failed to enforce schema of data '{"Moulins_temp":15.63,"Moulins_humidity":65.0,"Moulins_dew_point":9.08,"Moulins_clouds":92.0,"Moulins_wind_speed":2.52,"Moulins_wind_deg":180.0,"Aurillac_temp":13.99,"Aurillac_humidity":67.0,"Aurillac_dew_point":7.97,"Aurillac_clouds":100.0,"Aurillac_wind_speed":6.17,"Aurillac_wind_deg":140.0,"Saint-\u00c9tienne_temp":15.89,"Saint-\u00c9tienne_humidity":45.0,"Saint-\u00c9tienne_dew_point":3.99,"Saint-\u00c9tienne_clouds":0.0,"Saint-\u00c9tienne_wind_speed":6.17,"Saint-\u00c9tienne_wind_deg":180.0,"Annecy_temp":13.07,"Annecy_humidity":67.0,"Annecy_dew_point":7.09,"Annecy_clouds":0.0,"Annecy_wind_speed":2.06,"Annecy_wind_deg":90.0,"Nyons_temp":20.32,"Nyons_humidity":43.0,"Nyons_dew_point":7.34,"Nyons_clouds":27.0,"Nyons_wind_speed":2.54,"Nyons_wind_deg":145.0,"10cm":165.0,"K index Planetary":3.5}' with schema '['Moulins_temp': double (required), 'Moulins_humidity': long (required), 'Moulins_dew_point': double (required), 'Moulins_clouds'